# BEE Score Analysis: GPT-2 vs Medical-GPT2 on PubMed-200K-RCT

This notebook replicates the BEE (Bias Embedding Evaluation) fingerprinting experiment **without any fine-tuning**:

- **Target model**: `devmanpreet/Medical-GPT2-Classifier` — already fine-tuned on medical text; used as-is.
- **Reference model**: `openai-community/gpt2` — base GPT-2; we fit a sklearn LogisticRegression on its **frozen** embeddings (no PyTorch training).
- **Dataset**: `pietrolesci/pubmed-200k-rct` — PubMed 200K RCT abstracts with section labels.

**Hypothesis**: If the Medical-GPT2-Classifier was trained on data similar to PubMed-200K-RCT, its BEE fingerprint (keyword-class bias scores) should correlate with the BEE fingerprint of a probe trained on PubMed data from scratch.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers datasets torch yake nltk scikit-learn matplotlib seaborn tqdm openpyxl

## 2. Reproducibility — Set All Seeds

In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42

# Required for CUDA deterministic ops
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True)
    print("Deterministic algorithms: ENABLED")
except Exception as e:
    print(f"Warning — deterministic algorithms not fully available: {e}")

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"SEED = {SEED}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Imports

In [ ]:
import gc
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2Model,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_sim

import nltk
import yake

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 4. Configuration

In [ ]:
# --------------------------------------------------------------------------
# Model identifiers
# --------------------------------------------------------------------------
MODEL_BASE_ID   = "openai-community/gpt2"
MODEL_TARGET_ID = "devmanpreet/Medical-GPT2-Classifier"

# --------------------------------------------------------------------------
# Dataset
# --------------------------------------------------------------------------
DATASET_NAME = "pietrolesci/pubmed-200k-rct"

# --------------------------------------------------------------------------
# Tokenisation
# --------------------------------------------------------------------------
MAX_LENGTH = 128

# --------------------------------------------------------------------------
# Keyword extraction (YAKE)
# --------------------------------------------------------------------------
YAKE_TOP_KEYWORDS = 200    # keywords to extract
N_KEYWORD_TEXTS   = 50_000 # samples used for keyword extraction

# --------------------------------------------------------------------------
# Sklearn probe
# --------------------------------------------------------------------------
N_PROBE_SAMPLES = 20_000   # samples used to fit the LogisticRegression probe
LR_MAX_ITER     = 2000

# --------------------------------------------------------------------------
# BEE visualisation
# --------------------------------------------------------------------------
TOP_K_DISPLAY   = 15   # keywords shown per class heatmap
TOP_K_OVERLAP   = 20   # keywords used for Jaccard overlap comparison

print("Configuration loaded.")
print(f"  Base model   : {MODEL_BASE_ID}")
print(f"  Target model : {MODEL_TARGET_ID}")
print(f"  Dataset      : {DATASET_NAME}")

## 5. Load Dataset

In [ ]:
print(f"Downloading: {DATASET_NAME}")
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print("\nFeatures:", raw_dataset['train'].features)
print("\nSample row:")
print(raw_dataset['train'][0])

## 6. Dataset Exploration & Preprocessing

In [ ]:
# Combine ALL labeled splits — train + validation + test
split_dfs = []
for split_name in raw_dataset.keys():
    df_split = raw_dataset[split_name].to_pandas()
    df_split['_split'] = split_name
    split_dfs.append(df_split)
    print(f"  {split_name:<12s}: {len(df_split):,} rows")

train_df = pd.concat(split_dfs, ignore_index=True)

print(f"\nCombined shape : {train_df.shape}")
print(f"Columns        : {train_df.columns.tolist()}")

In [ ]:
# --------------------------------------------------------------------------
# Detect text column dynamically
# --------------------------------------------------------------------------
sample = raw_dataset['train'][0]

if 'sentence' in sample:
    TEXT_COL = 'sentence'
elif 'text' in sample:
    TEXT_COL = 'text'
elif 'sentence_text' in sample:
    TEXT_COL = 'sentence_text'
else:
    TEXT_COL = next(
        (k for k, v in sample.items() if isinstance(v, str) and k not in ('label', 'labels', '_split', 'uid')),
        None
    )
    if TEXT_COL is None:
        raise ValueError(f"Cannot detect text column. Keys: {list(sample.keys())}")

# --------------------------------------------------------------------------
# Detect label column dynamically (handles 'label', 'labels', etc.)
# --------------------------------------------------------------------------
for candidate in ('label', 'labels'):
    if candidate in train_df.columns:
        LABEL_COL = candidate
        break
else:
    raise ValueError(f"Cannot find label column. Columns: {train_df.columns.tolist()}")

print(f"Text column  : '{TEXT_COL}'")
print(f"Label column : '{LABEL_COL}'")

# --------------------------------------------------------------------------
# Handle ClassLabel (integer) vs string labels
# --------------------------------------------------------------------------
label_feature = raw_dataset['train'].features[LABEL_COL]
print(f"Label feature type: {type(label_feature).__name__}")

if hasattr(label_feature, 'names'):
    int2str = {i: name for i, name in enumerate(label_feature.names)}
    train_df['label_str'] = train_df[LABEL_COL].map(int2str)
    LABEL_COL = 'label_str'
    print(f"Mapped ClassLabel integers → strings: {int2str}")

# Normalise to upper-case
train_df[LABEL_COL] = train_df[LABEL_COL].astype(str).str.upper().str.strip()

# Drop rows with missing text/label
train_df = train_df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
train_df[TEXT_COL] = train_df[TEXT_COL].astype(str)

# Encode labels
le_pubmed = LabelEncoder()
le_pubmed.fit(sorted(train_df[LABEL_COL].unique()))   # sort for determinism
train_df['label_id'] = le_pubmed.transform(train_df[LABEL_COL])

PUBMED_ID2LABEL = {i: label for i, label in enumerate(le_pubmed.classes_)}
PUBMED_LABEL2ID = {v: k for k, v in PUBMED_ID2LABEL.items()}
PUBMED_NUM_LABELS = len(PUBMED_ID2LABEL)

print(f"\nPubMed label mapping ({PUBMED_NUM_LABELS} classes):")
for idx, label in PUBMED_ID2LABEL.items():
    count = (train_df['label_id'] == idx).sum()
    print(f"  {idx}: {label:<20s} ({count:,} samples)")

## 7. Helper Functions

In [ ]:
def normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / np.maximum(norms, 1e-8)


def extract_keywords_from_text(texts: list, top: int = 200, seed: int = SEED) -> list:
    random.seed(seed)
    np.random.seed(seed)
    parts = []
    for t in tqdm(texts, desc="Building keyword corpus", unit="sent", leave=True):
        parts.append(str(t))
    joined = " ".join(parts)
    tqdm.write(f"  Running YAKE on {len(joined):,} chars → top {top} keywords...")
    extractor = yake.KeywordExtractor(lan="en", n=1, dedupLim=0.9, top=top, features=None)
    keywords = extractor.extract_keywords(joined)
    tqdm.write(f"  YAKE done — {len(keywords)} keywords extracted.")
    return [k[0] for k in keywords]


def embed_texts_with_gpt2model(texts, model, tokenizer, batch_size=32, max_len=MAX_LENGTH):
    model.eval()
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding", leave=False):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
            last_hidden = out.last_hidden_state
            seq_lens = enc['attention_mask'].sum(dim=1) - 1
            b_idx = torch.arange(last_hidden.size(0), device=device)
            embs = last_hidden[b_idx, seq_lens]
        all_embs.append(embs.cpu().float().numpy())
    return np.vstack(all_embs)


def embed_texts_with_seq_clf(texts, model, tokenizer, batch_size=32, max_len=MAX_LENGTH):
    model.eval()
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding", leave=False):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
            last_hidden = out.hidden_states[-1]
            seq_lens = enc['attention_mask'].sum(dim=1) - 1
            b_idx = torch.arange(last_hidden.size(0), device=device)
            embs = last_hidden[b_idx, seq_lens]
        all_embs.append(embs.cpu().float().numpy())
    return np.vstack(all_embs)


def compute_bee_scores(
    classifier_weights_norm: np.ndarray,
    keyword_embeddings_norm: np.ndarray,
    keywords: list,
    id2label: dict,
):
    """
    BEE SC scores — paper Eq. 2:
        s+_{k,i} = w*ᵀ_k M(c_i)  −  min_{k'} w*ᵀ_{k'} M(c_i)

    Returns
    -------
    bee_df    : DataFrame, one row per keyword, one column per class (s+_{k,i} ≥ 0).
    sorted_sc : (num_classes, num_keywords) SC scores, same sort order as bee_df.
    raw_sims  : (num_classes, num_keywords) raw cosine similarities (unsorted).
    """
    raw_sims  = classifier_weights_norm @ keyword_embeddings_norm.T
    min_sim   = raw_sims.min(axis=0, keepdims=True)
    sc_scores = raw_sims - min_sim                        # (K, N), all ≥ 0

    overall_bias    = sc_scores.max(axis=0)
    order           = np.argsort(overall_bias)[::-1]
    sorted_keywords = np.array(keywords)[order]
    sorted_bias     = overall_bias[order]
    sorted_sc       = sc_scores[:, order]

    bee_df = pd.DataFrame({'Keyword': sorted_keywords, 'Bias_Score': sorted_bias})
    for i, label in id2label.items():
        bee_df[label] = sorted_sc[i]

    return bee_df, sorted_sc, raw_sims


def visualize_bee_results(
    bee_df: pd.DataFrame,
    id2label: dict,
    title_prefix: str = "",
    top_k: int = 15,
):
    """
    One heatmap per class.  Each heatmap shows the top_k keywords ranked
    by s+_{k,i} for that class (rows), with the SC score for that single
    class (one column).  Keywords are sorted highest → lowest.
    """
    for class_idx, class_label in id2label.items():
        # Top-k keywords for this class, sorted by their SC score descending
        top = (
            bee_df[['Keyword', class_label]]
            .nlargest(top_k, class_label)
            .reset_index(drop=True)
        )
        heatmap_data = top.set_index('Keyword')[[class_label]]

        fig, ax = plt.subplots(figsize=(3, max(4, top_k * 0.45)))
        sns.heatmap(
            heatmap_data, cmap="YlOrRd", vmin=0,
            annot=True, fmt=".2f", cbar=True, ax=ax,
        )
        ax.set_title(
            f"{title_prefix}\nClass: {class_label}  —  top {top_k} keywords",
            fontsize=11,
        )
        ax.set_xlabel("")
        ax.set_ylabel("Keyword")
        ax.set_xticklabels([class_label], rotation=0)
        plt.tight_layout()
        plt.show()


print("Helper functions defined.")

## 8. Target Model — `devmanpreet/Medical-GPT2-Classifier`

Loaded as-is. No fine-tuning performed.

In [ ]:
from transformers import GPT2ForSequenceClassification, GPT2Config
from huggingface_hub import hf_hub_download

print("=" * 60)
print(f"Loading TARGET model: {MODEL_TARGET_ID}")
print("=" * 60)

# ------------------------------------------------------------------
# Tokenizer: use base GPT-2 (identical vocab)
# ------------------------------------------------------------------
target_tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE_ID)
if target_tokenizer.pad_token is None:
    target_tokenizer.pad_token = target_tokenizer.eos_token
print(f"Tokenizer: {MODEL_BASE_ID}")

# ------------------------------------------------------------------
# Download raw .pth checkpoint (cached after first run)
# ------------------------------------------------------------------
pth_path = hf_hub_download(repo_id=MODEL_TARGET_ID,
                            filename="biofinetuned_partialEpoch1.pth")
ckpt = torch.load(pth_path, map_location="cpu", weights_only=False)
state_dict = ckpt if isinstance(ckpt, dict) else ckpt.state_dict()
print(f"Checkpoint: {len(state_dict)} keys loaded.")

# ------------------------------------------------------------------
# Detect architecture details from actual key names
# ------------------------------------------------------------------
n_layers = max(int(k.split('.')[1])
               for k in state_dict if k.startswith('trf_blocks.')) + 1

ff_second_idx = 2 if any('feedforw.layers.2' in k for k in state_dict) else 1

num_labels      = state_dict['out_head.weight'].shape[0]
print(f"Layers={n_layers}  ff_second_idx={ff_second_idx}  num_labels={num_labels}")

# ------------------------------------------------------------------
# Remap to HuggingFace GPT2ForSequenceClassification format
#
# Custom arch (nn.Linear)  →  HF GPT-2 (Conv1D)
#   weight shape: (out, in)  →  (in, out)  ∴ transpose every weight
#   bias / LayerNorm: no change
#
# Custom LayerNorm stores weight as 'scale_params' and bias as 'shift_params'
# Custom FF module is named 'feedforw' (not 'ff')
# Final norm uses 'scale_params' / 'shift_params' (not 'weight' / 'bias')
# ------------------------------------------------------------------
def remap_to_hf(sd, n_layers, ff_second_idx):
    hf = {}

    # Embeddings
    hf['transformer.wte.weight'] = sd['token_emb.weight']
    hf['transformer.wpe.weight'] = sd['pos_emb.weight']

    for n in range(n_layers):
        p = f'trf_blocks.{n}'

        # Q / K / V → combined c_attn (Conv1D: transpose each Linear weight)
        W_q = sd[f'{p}.attn.W_query.weight']
        W_k = sd[f'{p}.attn.W_key.weight']
        W_v = sd[f'{p}.attn.W_value.weight']
        b_q = sd[f'{p}.attn.W_query.bias']
        b_k = sd[f'{p}.attn.W_key.bias']
        b_v = sd[f'{p}.attn.W_value.bias']
        hf[f'transformer.h.{n}.attn.c_attn.weight'] = torch.cat([W_q.T, W_k.T, W_v.T], dim=1)  # (768, 2304)
        hf[f'transformer.h.{n}.attn.c_attn.bias']   = torch.cat([b_q, b_k, b_v], dim=0)         # (2304,)

        # Output projection
        hf[f'transformer.h.{n}.attn.c_proj.weight'] = sd[f'{p}.attn.out_proj.weight'].T
        hf[f'transformer.h.{n}.attn.c_proj.bias']   = sd[f'{p}.attn.out_proj.bias']

        # Layer norms  ('scale_params' → weight,  'shift_params' → bias)
        hf[f'transformer.h.{n}.ln_1.weight'] = sd[f'{p}.Layernorm1.scale_params']
        hf[f'transformer.h.{n}.ln_1.bias']   = sd[f'{p}.Layernorm1.shift_params']
        hf[f'transformer.h.{n}.ln_2.weight'] = sd[f'{p}.Layernorm2.scale_params']
        hf[f'transformer.h.{n}.ln_2.bias']   = sd[f'{p}.Layernorm2.shift_params']

        # Feed-forward  ('feedforw', not 'ff')
        hf[f'transformer.h.{n}.mlp.c_fc.weight']   = sd[f'{p}.feedforw.layers.0.weight'].T
        hf[f'transformer.h.{n}.mlp.c_fc.bias']     = sd[f'{p}.feedforw.layers.0.bias']
        hf[f'transformer.h.{n}.mlp.c_proj.weight'] = sd[f'{p}.feedforw.layers.{ff_second_idx}.weight'].T
        hf[f'transformer.h.{n}.mlp.c_proj.bias']   = sd[f'{p}.feedforw.layers.{ff_second_idx}.bias']

    # Final layer norm
    hf['transformer.ln_f.weight'] = sd['final_norm.scale_params']
    hf['transformer.ln_f.bias']   = sd['final_norm.shift_params']

    # Classification head (HF score has no bias — skip out_head.bias)
    hf['score.weight'] = sd['out_head.weight']

    return hf

hf_sd = remap_to_hf(state_dict, n_layers, ff_second_idx)
print(f"Remapped {len(hf_sd)} keys.")

# ------------------------------------------------------------------
# Build GPT2ForSequenceClassification shell and load weights
# ------------------------------------------------------------------
cfg = GPT2Config.from_pretrained(MODEL_BASE_ID)
cfg.num_labels   = num_labels
cfg.id2label     = {i: str(i) for i in range(num_labels)}
cfg.label2id     = {str(i): i for i in range(num_labels)}
cfg.pad_token_id = target_tokenizer.pad_token_id

target_model = GPT2ForSequenceClassification(cfg)
missing, unexpected = target_model.load_state_dict(hf_sd, strict=False)

print(f"load_state_dict — missing={len(missing)}  unexpected={len(unexpected)}")
if missing:    print(f"  Missing   : {missing}")
if unexpected: print(f"  Unexpected: {unexpected}")

target_model.to(device)
target_model.eval()

TARGET_CLF_HEAD    = target_model.score
TARGET_ID2LABEL    = target_model.config.id2label
TARGET_LABEL_NAMES = list(TARGET_ID2LABEL.values())

print(f"\nClassifier head : {TARGET_CLF_HEAD.weight.shape}")
print(f"num_labels      : {num_labels}")
print(f"Model has NaN   : {any(torch.isnan(p).any() for p in target_model.parameters())}")
print("Target model ready.")

## 9. Keyword Extraction from PubMed Data

In [ ]:
# Sample a fixed subset for keyword extraction (seeded for reproducibility)
rng_kw = np.random.default_rng(SEED)
all_texts = train_df[TEXT_COL].tolist()
kw_sample_idx = rng_kw.choice(len(all_texts), size=min(N_KEYWORD_TEXTS, len(all_texts)), replace=False)
sample_texts = [all_texts[i] for i in kw_sample_idx]

print(f"Extracting keywords from {len(sample_texts):,} / {len(all_texts):,} PubMed sentences...")
raw_keywords = extract_keywords_from_text(sample_texts, top=YAKE_TOP_KEYWORDS)

# Filter out class-name words and single characters
filter_words = {'background', 'objective', 'method', 'methods', 'result', 'results',
                'conclusion', 'conclusions', 'abstract'}
clean_keywords = [
    kw for kw in raw_keywords
    if kw.lower() not in filter_words and len(kw.strip()) > 1
]

print(f"Raw keywords  : {len(raw_keywords)}")
print(f"Clean keywords: {len(clean_keywords)}")
print(f"\nTop 20: {clean_keywords[:20]}")

## 10. BEE Analysis — Target Model (Medical-GPT2-Classifier)

In [ ]:
print("Computing BEE scores for TARGET model...")

# 1. Classifier weights
target_weights_raw  = TARGET_CLF_HEAD.weight.detach().cpu().float().numpy()  # (num_classes, H)
target_weights_norm = normalize_embeddings(target_weights_raw)
print(f"Classifier weight shape: {target_weights_raw.shape}")

# 2. Embed keywords using the target model's body
print("Embedding keywords with target model...")
target_kw_embs = embed_texts_with_seq_clf(
    clean_keywords, target_model, target_tokenizer, batch_size=64, max_len=32
)
target_kw_embs_norm = normalize_embeddings(target_kw_embs)
print(f"Keyword embedding shape: {target_kw_embs.shape}")

# 3. BEE SC scores  s+_{k,i} = sim(w_k, M(c_i)) - min_{k'} sim(w_{k'}, M(c_i))
target_bee_df, target_sc_sorted, target_raw_sims = compute_bee_scores(
    target_weights_norm, target_kw_embs_norm, clean_keywords, TARGET_ID2LABEL
)

print(f"\nTop {TOP_K_DISPLAY} BEE keywords — TARGET model:")
print(target_bee_df[['Keyword', 'Bias_Score']].head(TOP_K_DISPLAY).to_string(index=False))

In [ ]:
# ============================================================
# DIAGNOSTIC: Why are the heatmap columns almost identical?
# ============================================================
# Possible causes:
#   (A) Remapping bug  — transposing weights incorrectly produces
#       nearly parallel class weight vectors.
#   (B) Undertrained model — biofinetuned_partialEpoch1.pth was
#       only run for < 1 epoch, so class weights never diverged.
# ============================================================

print("=" * 65)
print("DIAGNOSTIC — Target Model BEE")
print("=" * 65)

# ----------------------------------------------------------
# 1. Pairwise cosine similarity between the 5 class weight
#    vectors.  If these are all ~ 1.0 the classifier head
#    never specialised → undertrained.
#    If they show diversity but BEE is still flat → embedding
#    extraction or similarity computation is the bug.
# ----------------------------------------------------------
pairwise_cls = sklearn_cosine_sim(target_weights_norm)   # (5, 5)
cls_labels   = TARGET_LABEL_NAMES

print("\n[1] Pairwise cosine similarity — TARGET class weight vectors:")
pairwise_df = pd.DataFrame(pairwise_cls,
                            index=cls_labels,
                            columns=cls_labels)
print(pairwise_df.round(4).to_string())

off_diag = pairwise_cls[np.triu_indices(len(cls_labels), k=1)]
print(f"\n  Off-diagonal stats → mean: {off_diag.mean():.4f}  "
      f"min: {off_diag.min():.4f}  max: {off_diag.max():.4f}")
print("  (near 1.0 = class vectors almost identical = undertrained/collapsed head)")

# ----------------------------------------------------------
# 2. BEE bias score statistics
# ----------------------------------------------------------
bs = target_bee_df['Bias_Score']
print(f"\n[2] Bias score stats:")
print(f"  min={bs.min():.6f}  max={bs.max():.6f}  "
      f"mean={bs.mean():.6f}  std={bs.std():.6f}")
print(f"  Non-zero entries: {(bs > 1e-6).sum()} / {len(bs)}")

# ----------------------------------------------------------
# 3. Per-class similarity range across keywords
# ----------------------------------------------------------
print("\n[3] Per-class similarity range over all keywords:")
for col in TARGET_LABEL_NAMES:
    col_vals = target_bee_df[col].values
    print(f"  Class '{col}':  min={col_vals.min():.4f}  "
          f"max={col_vals.max():.4f}  std={col_vals.std():.6f}")

# ----------------------------------------------------------
# 4. Quick inference sanity check — feed one sentence per
#    pubmed class and check if the model discriminates.
# ----------------------------------------------------------
print("\n[4] Inference sanity check (one sample per PubMed class):")
sample_rows = []
for cid in range(PUBMED_NUM_LABELS):
    rows = train_df[train_df['label_id'] == cid]
    if len(rows):
        sample_rows.append(rows.iloc[0][TEXT_COL])

sample_enc = target_tokenizer(
    sample_rows,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors='pt',
).to(device)

with torch.no_grad():
    logits = target_model(**sample_enc).logits.cpu().float()

pred_classes = logits.argmax(dim=-1).tolist()
print(f"  Logits (one row per sample):")
for i, (row, preds) in enumerate(zip(logits.numpy(), pred_classes)):
    true_label = PUBMED_ID2LABEL[i]
    pred_label = TARGET_ID2LABEL[preds]
    print(f"  Sample[{i}] true_pubmed='{true_label}'  "
          f"pred_target='{pred_label}'  logits={np.round(row, 3)}")

logit_std_across_classes = logits.std(dim=-1)
print(f"\n  Logit std per sample (should be > 0.1 for a non-collapsed model):")
for i, s in enumerate(logit_std_across_classes.tolist()):
    print(f"    Sample[{i}]: std = {s:.4f}")

# ----------------------------------------------------------
# 5. Check raw (un-normalised) classifier weight norms
# ----------------------------------------------------------
print("\n[5] L2 norm of each (raw) class weight vector:")
for i, name in enumerate(TARGET_LABEL_NAMES):
    n = float(np.linalg.norm(target_weights_raw[i]))
    print(f"  Class '{name}': L2 norm = {n:.6f}")

# ----------------------------------------------------------
# Summary verdict
# ----------------------------------------------------------
mean_off_diag = float(off_diag.mean())
print("\n" + "=" * 65)
if mean_off_diag > 0.99:
    print("VERDICT: Class weight vectors are nearly IDENTICAL (mean cos-sim "
          f"= {mean_off_diag:.4f}).")
    print("  → Likely cause: model was barely trained (partialEpoch1).")
    print("    The classification head initialised near-uniformly and never")
    print("    diverged enough for BEE to distinguish classes.")
    print("  → The remapping itself is probably correct; the model is the issue.")
elif mean_off_diag > 0.90:
    print(f"VERDICT: Class vectors moderately similar (mean cos-sim = {mean_off_diag:.4f}).")
    print("  → Check [3]: if per-class sim range is very small, the embedding")
    print("    space is collapsed.  Consider verifying the Conv1D transpose.")
else:
    print(f"VERDICT: Class vectors look distinct (mean cos-sim = {mean_off_diag:.4f}).")
    print("  → Bug is likely in BEE computation or embedding extraction,")
    print("    NOT in the weight remapping.")

In [ ]:
# Visualise
visualize_bee_results(
    target_bee_df, TARGET_ID2LABEL,
    title_prefix="Medical-GPT2-Classifier (Target)",
    top_k=TOP_K_DISPLAY,
)

## 11. Reference Model — `openai-community/gpt2` (frozen body + sklearn probe)

We extract embeddings from the **frozen** GPT-2 body, then fit a `LogisticRegression` on those embeddings.  
The LR coefficients serve as the reference classifier weights for BEE — **no PyTorch training** occurs.

In [ ]:
print("=" * 60)
print(f"Loading BASE model body: {MODEL_BASE_ID}")
print("=" * 60)

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE_ID)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

# Load only the transformer body (no classification head)
base_gpt2 = GPT2Model.from_pretrained(MODEL_BASE_ID, torch_dtype=torch.float32)
base_gpt2.to(device)
base_gpt2.eval()

# Sanity check
has_nan = any(torch.isnan(p).any() for p in base_gpt2.parameters())
print(f"Base GPT-2 has NaN: {has_nan}")

total_params = sum(p.numel() for p in base_gpt2.parameters())
print(f"Base GPT-2 parameters: {total_params:,} (ALL frozen — no training)")

In [ ]:
# --------------------------------------------------------------------------
# Sample a fixed subset of the training set for the probe (seeded)
# --------------------------------------------------------------------------
rng_probe = np.random.default_rng(SEED)
probe_idx = rng_probe.choice(len(train_df), size=min(N_PROBE_SAMPLES, len(train_df)), replace=False)
probe_df  = train_df.iloc[probe_idx].reset_index(drop=True)

print(f"Probe set size: {len(probe_df):,}")
print(f"Label distribution in probe set:")
print(probe_df[LABEL_COL].value_counts().to_string())

In [ ]:
# --------------------------------------------------------------------------
# Extract frozen embeddings from base GPT-2 for the probe samples
# --------------------------------------------------------------------------
print("Extracting frozen GPT-2 embeddings for probe samples...")

probe_texts  = probe_df[TEXT_COL].tolist()
probe_labels = probe_df['label_id'].to_numpy()

probe_embeddings = embed_texts_with_gpt2model(
    probe_texts, base_gpt2, base_tokenizer, batch_size=64, max_len=MAX_LENGTH
)

print(f"Probe embeddings shape : {probe_embeddings.shape}")
print(f"Probe labels shape     : {probe_labels.shape}")

In [ ]:
# --------------------------------------------------------------------------
# Fit sklearn LogisticRegression on frozen embeddings
# Coefficients shape: (num_classes, hidden_size) — used as BEE classifier weights
# --------------------------------------------------------------------------
print("Fitting LogisticRegression probe on frozen GPT-2 embeddings...")

lr_probe = LogisticRegression(
    random_state=SEED,
    max_iter=LR_MAX_ITER,
    solver='lbfgs',
    multi_class='multinomial',
    C=1.0,
    n_jobs=-1,
)
lr_probe.fit(probe_embeddings, probe_labels)

# Evaluate probe
probe_preds = lr_probe.predict(probe_embeddings)
probe_acc   = accuracy_score(probe_labels, probe_preds)
probe_f1    = f1_score(probe_labels, probe_preds, average='weighted')

print(f"\nProbe train accuracy : {probe_acc:.4f}")
print(f"Probe train F1       : {probe_f1:.4f}")
print(f"\nClassification report:")
print(classification_report(
    probe_labels, probe_preds,
    target_names=[PUBMED_ID2LABEL[i] for i in range(PUBMED_NUM_LABELS)]
))

# Classifier weights from LR: shape (num_classes, hidden_size)
base_weights_raw  = lr_probe.coef_.astype(np.float32)   # (num_classes, H)
base_weights_norm = normalize_embeddings(base_weights_raw)
print(f"\nLR coefficient shape (= classifier weights): {base_weights_raw.shape}")

In [ ]:
# --------------------------------------------------------------------------
# Embed keywords using the frozen base GPT-2 body
# --------------------------------------------------------------------------
print("Embedding keywords with base GPT-2 body...")

base_kw_embs = embed_texts_with_gpt2model(
    clean_keywords, base_gpt2, base_tokenizer, batch_size=64, max_len=32
)
base_kw_embs_norm = normalize_embeddings(base_kw_embs)
print(f"Keyword embedding shape: {base_kw_embs.shape}")

# --------------------------------------------------------------------------
# BEE SC scores  s+_{k,i} = sim(w_k, M(c_i)) - min_{k'} sim(w_{k'}, M(c_i))
# --------------------------------------------------------------------------
base_bee_df, base_sc_sorted, base_raw_sims = compute_bee_scores(
    base_weights_norm, base_kw_embs_norm, clean_keywords, PUBMED_ID2LABEL
)

print(f"\nTop {TOP_K_DISPLAY} BEE keywords — REFERENCE model:")
print(base_bee_df[['Keyword', 'Bias_Score']].head(TOP_K_DISPLAY).to_string(index=False))

In [ ]:
# Visualise
visualize_bee_results(
    base_bee_df, PUBMED_ID2LABEL,
    title_prefix="GPT-2 Base + sklearn Probe (Reference)",
    top_k=TOP_K_DISPLAY,
)

## 12. ABR — Asymmetric Bias Residual

**Intuition**: BEE uses *classifier weights* as the class representation. ABR instead uses the *empirical class centroid* — the mean embedding of actual dataset samples. It measures how much the keyword direction is structurally embedded in the class's sample distribution.

**Formula** (per keyword `k`, class centroid `c`):

$$\mathbf{c}_i^{\perp} = \mathbf{c}_i - \frac{\mathbf{c}_i \cdot \mathbf{k}}{\|\mathbf{k}\|^2}\,\mathbf{k}, \qquad \mathbf{c}^{\perp} = \frac{1}{N}\sum_i \mathbf{c}_i^{\perp}, \qquad \text{ABR}(\mathbf{k}, \mathbf{c}) = \frac{\|\mathbf{c} - \mathbf{c}^{\perp}\|}{\|\mathbf{c}\|}$$

Because projection is linear this simplifies to `|cos(c, k)|`, but the centroid `c` is computed from actual sample embeddings (data-driven), not from classifier weights.

**Contrastive ABR**:
$$\text{cABR}(\mathbf{k}, c) = \frac{\text{ABR}_{\text{target}}(\mathbf{k}, c)}{\text{ABR}_{\text{reference}}(\mathbf{k}, c)}$$
A ratio >> 1 means the target model's embedding space contains a stronger association between the keyword and the class than the reference model — a membership signal.

In [ ]:
N_CENTROID_SAMPLES = 2000   # samples per class used to estimate the centroid


def compute_class_centroids(train_df, text_col, label_id_col, embed_fn,
                             model, tokenizer, id2label,
                             n_per_class=N_CENTROID_SAMPLES, batch_size=64):
    centroids = {}
    for class_id, class_label in id2label.items():
        class_df = train_df[train_df[label_id_col] == class_id]
        n = min(n_per_class, len(class_df))
        sample_texts = class_df.sample(n=n, random_state=SEED)[text_col].tolist()
        print(f"  Embedding {n:,} samples for class '{class_label}'...")
        embs = embed_fn(sample_texts, model, tokenizer,
                        batch_size=batch_size, max_len=MAX_LENGTH)
        centroids[class_label] = embs.mean(axis=0)
    return centroids


def compute_abr_scores(keyword_embeddings, keywords, centroids, eps=1e-8):
    class_labels = list(centroids.keys())
    n_kw  = keyword_embeddings.shape[0]
    n_cls = len(class_labels)

    if len(keywords) != n_kw:
        raise ValueError(
            f"Length mismatch: {len(keywords)} keywords but "
            f"{n_kw} keyword embeddings. Re-run the embedding cell(s)."
        )

    c_vecs = np.stack([centroids[cl] for cl in class_labels], axis=0)
    k_norm = keyword_embeddings / (np.linalg.norm(keyword_embeddings, axis=1, keepdims=True) + eps)
    c_norm = c_vecs / (np.linalg.norm(c_vecs, axis=1, keepdims=True) + eps)

    raw_abr = np.abs(k_norm @ c_norm.T)                       # (N_kw, K_cls)

    d_abr = np.empty_like(raw_abr)
    for j in range(n_cls):
        other = [i for i in range(n_cls) if i != j]
        d_abr[:, j] = raw_abr[:, j] - raw_abr[:, other].max(axis=1)

    abr_df = pd.DataFrame(raw_abr, columns=class_labels)
    abr_df.insert(0, 'Keyword', keywords)

    dabr_cols = [f'dABR_{cl}' for cl in class_labels]
    abr_df = pd.concat([abr_df, pd.DataFrame(d_abr, columns=dabr_cols)], axis=1)
    abr_df['Max_ABR']  = abr_df[class_labels].max(axis=1)
    abr_df['Max_dABR'] = abr_df[dabr_cols].max(axis=1)
    abr_df = abr_df.sort_values('Max_dABR', ascending=False).reset_index(drop=True)
    return abr_df


def visualize_abr_results(abr_df, class_labels, title_prefix="", top_k=15):
    """
    One heatmap per class. Keywords SELECTED by dABR (most uniquely
    associated with that class), then SORTED by raw ABR descending
    so the heatmap reads top-to-bottom from strongest to weakest.
    """
    for cls_label in class_labels:
        dcol = f'dABR_{cls_label}'
        top  = (
            abr_df[['Keyword', cls_label, dcol]]
            .nlargest(top_k, dcol)          # select by discriminative score
            .sort_values(cls_label, ascending=False)   # display sorted by raw ABR
            .reset_index(drop=True)
        )
        heatmap_data = top.set_index('Keyword')[[cls_label]]

        fig, ax = plt.subplots(figsize=(3, max(4, top_k * 0.45)))
        sns.heatmap(heatmap_data, cmap="Blues", vmin=0, vmax=1,
                    annot=True, fmt=".3f", cbar=True, ax=ax)
        ax.set_title(
            f"{title_prefix}\nABR — Class: {cls_label} — top {top_k}",
            fontsize=11,
        )
        ax.set_xlabel("")
        ax.set_ylabel("Keyword")
        ax.set_xticklabels([cls_label], rotation=0)
        plt.tight_layout()
        plt.show()


print("ABR helper functions defined.")

In [ ]:
# -----------------------------------------------------------------------
# Target model — class centroids + ABR scores
# Uses PubMed class labels (label_id) to group samples.
# Embeddings extracted via the target model's body.
# -----------------------------------------------------------------------
print("Computing TARGET model class centroids from PubMed samples...")
target_centroids = compute_class_centroids(
    train_df, TEXT_COL, 'label_id',
    embed_texts_with_seq_clf, target_model, target_tokenizer,
    PUBMED_ID2LABEL,
)

print("\nComputing TARGET ABR scores...")
target_abr_df = compute_abr_scores(
    target_kw_embs,      # raw keyword embeddings from target model
    clean_keywords,
    target_centroids,
)
print(f"Target ABR computed. Shape: {target_abr_df.shape}")
print(f"\nTop 5 keywords by max ABR — TARGET:")
print(target_abr_df[['Keyword', 'Max_ABR']].head(5).to_string(index=False))

print("\nVisualising TARGET ABR (one heatmap per class)...")
visualize_abr_results(
    target_abr_df,
    class_labels=list(target_centroids.keys()),
    title_prefix="Medical-GPT2-Classifier (Target)",
    top_k=TOP_K_DISPLAY,
)

In [ ]:
# -----------------------------------------------------------------------
# Reference model — class centroids + ABR scores
# Embeddings extracted via frozen base GPT-2 body.
# -----------------------------------------------------------------------
print("Computing REFERENCE model class centroids from PubMed samples...")
base_centroids = compute_class_centroids(
    train_df, TEXT_COL, 'label_id',
    embed_texts_with_gpt2model, base_gpt2, base_tokenizer,
    PUBMED_ID2LABEL,
)

print("\nComputing REFERENCE ABR scores...")
base_abr_df = compute_abr_scores(
    base_kw_embs,        # raw keyword embeddings from base GPT-2
    clean_keywords,
    base_centroids,
)
print(f"Reference ABR computed. Shape: {base_abr_df.shape}")
print(f"\nTop 5 keywords by max ABR — REFERENCE:")
print(base_abr_df[['Keyword', 'Max_ABR']].head(5).to_string(index=False))

print("\nVisualising REFERENCE ABR (one heatmap per class)...")
visualize_abr_results(
    base_abr_df,
    class_labels=list(base_centroids.keys()),
    title_prefix="GPT-2 Base + sklearn Probe (Reference)",
    top_k=TOP_K_DISPLAY,
)

In [ ]:
# -----------------------------------------------------------------------
# Contrastive ABR  =  ABR_target / ABR_reference
#
# For each (keyword, class) pair, how much stronger is the target model's
# association compared to the reference?  Ratio > 1 is a membership signal.
# -----------------------------------------------------------------------

# Align both ABR DataFrames on keyword order
target_abr_kw = target_abr_df.set_index('Keyword')
base_abr_kw   = base_abr_df.set_index('Keyword')

shared_kws     = sorted(set(target_abr_kw.index) & set(base_abr_kw.index))
pubmed_classes = list(PUBMED_ID2LABEL.values())

cabr_rows = []
for kw in shared_kws:
    row = {'Keyword': kw}
    for cls in pubmed_classes:
        t_val = target_abr_kw.at[kw, cls] if cls in target_abr_kw.columns else 0.0
        b_val = base_abr_kw.at[kw, cls]   if cls in base_abr_kw.columns   else 0.0
        # Avoid division by zero; cap ratio at 10 for display stability
        if b_val < 1e-6:
            row[f'cABR_{cls}'] = float('nan')
        else:
            row[f'cABR_{cls}'] = min(t_val / b_val, 10.0)
    cabr_rows.append(row)

cabr_df = pd.DataFrame(cabr_rows)
cabr_cols = [f'cABR_{cls}' for cls in pubmed_classes]
cabr_df['Max_cABR'] = cabr_df[cabr_cols].max(axis=1)
cabr_df = cabr_df.sort_values('Max_cABR', ascending=False).reset_index(drop=True)

print("Contrastive ABR computed.")
print(f"Shape: {cabr_df.shape}")
print(f"\nTop 10 keywords by max cABR (membership signal):")
print(cabr_df[['Keyword', 'Max_cABR'] + cabr_cols].head(10).round(3).to_string(index=False))

# --- Visualise: one heatmap per class ---
for cls in pubmed_classes:
    col = f'cABR_{cls}'
    top = (
        cabr_df[['Keyword', col]]
        .dropna()
        .nlargest(TOP_K_DISPLAY, col)
        .reset_index(drop=True)
    )
    heatmap_data = top.set_index('Keyword')[[col]]

    fig, ax = plt.subplots(figsize=(3, max(4, TOP_K_DISPLAY * 0.45)))
    sns.heatmap(
        heatmap_data, cmap="RdYlGn", center=1.0,
        annot=True, fmt=".2f", cbar=True, ax=ax,
    )
    ax.set_title(
        f"cABR  —  Class: {cls}  —  top {TOP_K_DISPLAY}\n"
        f"(>1 = target stronger, <1 = reference stronger)",
        fontsize=10,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Keyword")
    ax.set_xticklabels([col], rotation=0)
    plt.tight_layout()
    plt.show()

## 13. Full BEE Tables (All Keywords, Alphabetical)

In [ ]:
for bee_df, model_name in [
    (target_bee_df, f"Medical-GPT2-Classifier — classes: {TARGET_LABEL_NAMES}"),
    (base_bee_df,   f"GPT-2 Base + sklearn Probe — classes: {list(PUBMED_ID2LABEL.values())}"),
]:
    sorted_alpha = bee_df.sort_values('Keyword', ascending=True).reset_index(drop=True)
    label_cols = [c for c in sorted_alpha.columns if c not in ('Keyword', 'Bias_Score')]

    print(f"\n{'='*80}")
    print(f"BEE TABLE — {model_name} ({len(sorted_alpha)} keywords, A-Z)")
    print(f"{'='*80}")

    styled = (
        sorted_alpha.style
        .background_gradient(subset=label_cols, cmap="coolwarm", axis=None)
        .background_gradient(subset=["Bias_Score"], cmap="YlOrRd")
        .format({c: "{:.4f}" for c in label_cols + ["Bias_Score"]})
        .set_caption(model_name)
    )
    try:
        from IPython.display import display
        display(styled)
    except Exception:
        print(sorted_alpha.to_string())

## 14. Comparison of BEE Fingerprints

Since the two models may have **different numbers of classes**, we compare them using:
1. **Bias-score vector** — one scalar per keyword, model-agnostic.
2. **Top-K keyword overlap** — Jaccard index.
3. **Scatter plot + histogram** of bias scores.

In [ ]:
# Align both DataFrames on the shared keyword list
all_keywords_sorted = sorted(set(clean_keywords))

def build_bias_vector(bee_df: pd.DataFrame, keyword_list: list) -> np.ndarray:
    """Return bias score for each keyword in keyword_list (0 if missing)."""
    kw2score = dict(zip(bee_df['Keyword'], bee_df['Bias_Score']))
    return np.array([kw2score.get(kw, 0.0) for kw in keyword_list], dtype=np.float32)

target_bias_vec = build_bias_vector(target_bee_df, all_keywords_sorted)
base_bias_vec   = build_bias_vector(base_bee_df,   all_keywords_sorted)

# Cosine similarity between BEE bias vectors
cos_sim = sklearn_cosine_sim(
    target_bias_vec.reshape(1, -1),
    base_bias_vec.reshape(1, -1),
)[0, 0]

# Pearson correlation
pearson_r = float(np.corrcoef(target_bias_vec, base_bias_vec)[0, 1])

print("=" * 60)
print("BEE FINGERPRINT COMPARISON")
print("=" * 60)
print(f"  Cosine similarity (bias vectors) : {cos_sim:.4f}")
print(f"  Pearson correlation              : {pearson_r:.4f}")
print()
print("Interpretation:")
print("  High similarity → Medical-GPT2 bias fingerprint matches PubMed probe")
print("  Low similarity  → Models biased toward different vocabularies")

In [ ]:
# --------------------------------------------------------------------------
# Build aligned per-class similarity arrays for both models.
#
# Problem: target_bee_df and base_bee_df are each sorted by their own
# bias score (descending), so their rows are in different keyword orders.
# Comparing them column-by-column would silently mix up keywords.
#
# Solution: re-index every column onto all_keywords_sorted (alphabetical,
# shared between both models) and store everything in fingerprint_store.
# Missing keywords are filled with 0.0.
# --------------------------------------------------------------------------

def build_class_vector(bee_df: pd.DataFrame, keyword_list: list, class_col: str) -> np.ndarray:
    """Return per-class similarity for each keyword in keyword_list (0 if missing)."""
    kw2sim = dict(zip(bee_df['Keyword'], bee_df[class_col]))
    return np.array([kw2sim.get(kw, 0.0) for kw in keyword_list], dtype=np.float32)


# -- Target model: one array per class, shape (N_keywords,) --
target_class_cols = [c for c in target_bee_df.columns if c not in ('Keyword', 'Bias_Score')]
target_aligned_sims = {
    col: build_class_vector(target_bee_df, all_keywords_sorted, col)
    for col in target_class_cols
}

# -- Reference model: one array per class, shape (N_keywords,) --
base_class_cols = [c for c in base_bee_df.columns if c not in ('Keyword', 'Bias_Score')]
base_aligned_sims = {
    col: build_class_vector(base_bee_df, all_keywords_sorted, col)
    for col in base_class_cols
}

# --------------------------------------------------------------------------
# fingerprint_store — the single source of truth for all future metrics.
#
# Layout
# ------
# fingerprint_store
# ├── 'keywords'          list[str]           shared keyword index (alphabetical)
# ├── 'target'
# │   ├── 'bias_vec'      np.array (N,)       max-min gap per keyword
# │   ├── 'class_sims'    dict[str → (N,)]    per-class cosine similarity per keyword
# │   ├── 'weights_norm'  np.array (C_t, H)   normalised classifier weights
# │   └── 'kw_embs_norm'  np.array (N, H)     normalised keyword embeddings
# └── 'reference'
#     ├── 'bias_vec'      np.array (N,)
#     ├── 'class_sims'    dict[str → (N,)]
#     ├── 'weights_norm'  np.array (C_r, H)
#     └── 'kw_embs_norm'  np.array (N, H)
# --------------------------------------------------------------------------

# Re-index keyword embeddings to match all_keywords_sorted
# (clean_keywords is in YAKE order; all_keywords_sorted is alphabetical)
_kw_to_idx = {kw: i for i, kw in enumerate(clean_keywords)}

target_kw_embs_aligned = np.array([
    target_kw_embs_norm[_kw_to_idx[kw]] if kw in _kw_to_idx else np.zeros(target_kw_embs_norm.shape[1])
    for kw in all_keywords_sorted
], dtype=np.float32)

base_kw_embs_aligned = np.array([
    base_kw_embs_norm[_kw_to_idx[kw]] if kw in _kw_to_idx else np.zeros(base_kw_embs_norm.shape[1])
    for kw in all_keywords_sorted
], dtype=np.float32)

fingerprint_store = {
    'keywords': all_keywords_sorted,          # shared index — same order for every array below
    'target': {
        'bias_vec':     target_bias_vec,       # (N,)   — aligned on keywords
        'class_sims':   target_aligned_sims,   # {class_name: (N,)}
        'weights_norm': target_weights_norm,   # (C_t, H)
        'kw_embs_norm': target_kw_embs_aligned,# (N, H) — aligned on keywords
    },
    'reference': {
        'bias_vec':     base_bias_vec,         # (N,)   — aligned on keywords
        'class_sims':   base_aligned_sims,     # {class_name: (N,)}
        'weights_norm': base_weights_norm,     # (C_r, H)
        'kw_embs_norm': base_kw_embs_aligned,  # (N, H) — aligned on keywords
    },
}

# --------------------------------------------------------------------------
# Quick sanity check
# --------------------------------------------------------------------------
N = len(all_keywords_sorted)
print("fingerprint_store — shapes:")
print(f"  keywords              : {N} entries")
print(f"  target  bias_vec      : {fingerprint_store['target']['bias_vec'].shape}")
print(f"  target  class_sims    : {list(fingerprint_store['target']['class_sims'].keys())}")
print(f"  target  weights_norm  : {fingerprint_store['target']['weights_norm'].shape}")
print(f"  target  kw_embs_norm  : {fingerprint_store['target']['kw_embs_norm'].shape}")
print(f"  reference bias_vec    : {fingerprint_store['reference']['bias_vec'].shape}")
print(f"  reference class_sims  : {list(fingerprint_store['reference']['class_sims'].keys())}")
print(f"  reference weights_norm: {fingerprint_store['reference']['weights_norm'].shape}")
print(f"  reference kw_embs_norm: {fingerprint_store['reference']['kw_embs_norm'].shape}")
print()
print("All arrays share the same keyword index — safe to compare element-wise.")
print()
print("Example — add any metric below, e.g.:")
print("  from scipy.stats import spearmanr, kendalltau")
print("  spearmanr(fingerprint_store['target']['bias_vec'],")
print("            fingerprint_store['reference']['bias_vec'])")

### Central Fingerprint Store
All BEE data aligned on `all_keywords_sorted` and collected into `fingerprint_store`.  
Add any future metric in a new cell below — every array you need is already here.

In [ ]:
# --------------------------------------------------------------------------
# Top-K keyword overlap (Jaccard index)
# --------------------------------------------------------------------------
target_top_k = set(target_bee_df['Keyword'].head(TOP_K_OVERLAP).tolist())
base_top_k   = set(base_bee_df['Keyword'].head(TOP_K_OVERLAP).tolist())

overlap = target_top_k & base_top_k
jaccard = len(overlap) / len(target_top_k | base_top_k)

print(f"Top-{TOP_K_OVERLAP} keyword overlap:")
print(f"  Target model top-{TOP_K_OVERLAP}   : {sorted(target_top_k)}")
print(f"  Reference model top-{TOP_K_OVERLAP}: {sorted(base_top_k)}")
print(f"  Shared keywords ({len(overlap)})   : {sorted(overlap)}")
print(f"  Jaccard index            : {jaccard:.4f}")

## 12b. NOS — Neighborhood Overlap Score

**Intuition**: For each keyword, find its K nearest neighbors in both the target and reference embedding spaces. If the model was trained on data containing these keywords, the local neighborhood structure should differ from the reference.

**Formula**:
$$\text{NOS}(k) = \frac{|\text{KNN}_{\text{target}}(k) \cap \text{KNN}_{\text{ref}}(k)|}{K}$$

- NOS = 1.0 → identical neighborhoods (no membership signal)
- NOS ≈ 0.0 → completely different neighbors (strong signal that target reorganised the embedding space around this keyword)

In [ ]:
def compute_nos_scores(target_kw_embs_norm, base_kw_embs_norm, keywords, k=50):
    """
    Neighborhood Overlap Score (NOS) for each keyword.

    For each keyword i, find its K nearest neighbors (by cosine similarity)
    among all other keywords in both embedding spaces, then compute the
    fraction of shared neighbors.

    Parameters
    ----------
    target_kw_embs_norm : (N, H) normalised keyword embeddings from target model
    base_kw_embs_norm   : (N, H) normalised keyword embeddings from reference model
    keywords            : list of N keyword strings
    k                   : number of neighbors to consider

    Returns
    -------
    nos_df : DataFrame with columns [Keyword, NOS, NOS_rank]
    """
    n = len(keywords)
    k = min(k, n - 1)  # can't have more neighbors than keywords - 1

    # Cosine similarity matrices (N x N)
    sim_target = target_kw_embs_norm @ target_kw_embs_norm.T
    sim_base   = base_kw_embs_norm @ base_kw_embs_norm.T

    nos_values = np.empty(n, dtype=np.float32)
    for i in range(n):
        # Exclude self-similarity by setting diagonal to -inf
        t_sims = sim_target[i].copy()
        b_sims = sim_base[i].copy()
        t_sims[i] = -np.inf
        b_sims[i] = -np.inf

        # Top-K neighbor indices
        t_neighbors = set(np.argpartition(t_sims, -k)[-k:])
        b_neighbors = set(np.argpartition(b_sims, -k)[-k:])

        nos_values[i] = len(t_neighbors & b_neighbors) / k

    nos_df = pd.DataFrame({
        'Keyword': keywords,
        'NOS': nos_values,
    })
    nos_df = nos_df.sort_values('NOS', ascending=True).reset_index(drop=True)
    nos_df['NOS_rank'] = range(1, n + 1)
    return nos_df


print("NOS helper function defined.")

In [ ]:
# -----------------------------------------------------------------------
# Compute NOS using aligned keyword embeddings from fingerprint_store
# -----------------------------------------------------------------------
print("Computing NOS scores (K=50 neighbors)...")

nos_df = compute_nos_scores(
    fingerprint_store['target']['kw_embs_norm'],
    fingerprint_store['reference']['kw_embs_norm'],
    fingerprint_store['keywords'],
    k=50,
)

print(f"NOS computed. Shape: {nos_df.shape}")
print(f"\nNOS statistics:")
print(f"  mean={nos_df['NOS'].mean():.4f}  std={nos_df['NOS'].std():.4f}")
print(f"  min={nos_df['NOS'].min():.4f}   max={nos_df['NOS'].max():.4f}")
print(f"\nTop 15 keywords with LOWEST NOS (most divergent neighborhoods):")
print(nos_df[['Keyword', 'NOS']].head(15).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# NOS Visualisation — per-class heatmap of keywords with lowest NOS
#
# Class assignment: each keyword is assigned to the class where it has
# the highest BEE SC score (from the target model).
# -----------------------------------------------------------------------
pubmed_classes = list(PUBMED_ID2LABEL.values())

# Assign each keyword to its dominant class using target BEE SC scores
kw_to_class = {}
for _, row in target_bee_df.iterrows():
    kw = row['Keyword']
    class_scores = {cls: row[cls] for cls in target_bee_df.columns
                    if cls not in ('Keyword', 'Bias_Score') and cls in pubmed_classes}
    if class_scores:
        kw_to_class[kw] = max(class_scores, key=class_scores.get)

nos_df['Dominant_Class'] = nos_df['Keyword'].map(kw_to_class)

# Per-class heatmaps
for cls in pubmed_classes:
    cls_nos = nos_df[nos_df['Dominant_Class'] == cls].head(TOP_K_DISPLAY)
    if len(cls_nos) == 0:
        continue
    heatmap_data = cls_nos.set_index('Keyword')[['NOS']]

    fig, ax = plt.subplots(figsize=(3, max(4, len(cls_nos) * 0.45)))
    sns.heatmap(
        heatmap_data, cmap="YlGnBu_r", vmin=0, vmax=1,
        annot=True, fmt=".3f", cbar=True, ax=ax,
    )
    ax.set_title(
        f"NOS — Class: {cls} — top {TOP_K_DISPLAY}\n"
        f"(lower = more divergent neighborhoods)",
        fontsize=10,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Keyword")
    plt.tight_layout()
    plt.show()

# Summary histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(nos_df['NOS'], bins=30, color='teal', edgecolor='white', alpha=0.8)
ax.axvline(nos_df['NOS'].mean(), color='red', linestyle='--', label=f"mean={nos_df['NOS'].mean():.3f}")
ax.set_title("NOS Distribution Across All Keywords", fontsize=12)
ax.set_xlabel("NOS (0=fully divergent, 1=identical neighborhoods)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 12c. EDP — Embedding Drift Under Perturbation

**Intuition**: If a model memorised a keyword during training, its internal representation should be more sensitive to character-level perturbations (typos). A model that merely knows the word generically should be robust to minor misspellings.

**Formula**:
$$\text{EDP}(k, \mathcal{M}) = \frac{\|E_\mathcal{M}(k) - E_\mathcal{M}(\tilde{k})\|}{\|E_\mathcal{M}(k)\|}$$

where $\tilde{k}$ is a deterministic character-level perturbation of keyword $k$.

**Contrastive**:
$$\text{cEDP}(k) = \frac{\text{EDP}_{\text{target}}(k)}{\text{EDP}_{\text{ref}}(k)}$$

Ratio > 1 → target is more sensitive to perturbation → memorisation signal.

In [ ]:
def generate_perturbations(keywords, seed=SEED):
    """
    Generate one deterministic character-level perturbation per keyword.

    Strategy (deterministic, seeded):
      - Words with len >= 4: swap two adjacent characters at a seeded position
      - Words with len == 3: delete middle character
      - Words with len <= 2: duplicate the first character

    Returns list of perturbed strings in the same order as `keywords`.
    """
    rng = np.random.default_rng(seed)
    perturbed = []
    for kw in keywords:
        if len(kw) >= 4:
            # Swap two adjacent characters (not first/last to keep recognisable)
            pos = int(rng.integers(1, len(kw) - 2))
            chars = list(kw)
            chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]
            perturbed.append(''.join(chars))
        elif len(kw) == 3:
            perturbed.append(kw[0] + kw[2])  # delete middle char
        else:
            perturbed.append(kw[0] + kw)  # duplicate first char
    return perturbed


def compute_edp_scores(orig_embs, perturbed_embs, keywords, eps=1e-8):
    """
    Embedding Drift under Perturbation.

    EDP(k) = ||embed(k) - embed(perturbed_k)|| / ||embed(k)||

    Parameters
    ----------
    orig_embs      : (N, H) raw embeddings of original keywords
    perturbed_embs : (N, H) raw embeddings of perturbed keywords
    keywords       : list of N keyword strings

    Returns
    -------
    edp_df : DataFrame with columns [Keyword, EDP]
    """
    diff_norms = np.linalg.norm(orig_embs - perturbed_embs, axis=1)
    orig_norms = np.linalg.norm(orig_embs, axis=1)
    edp_values = diff_norms / np.maximum(orig_norms, eps)

    edp_df = pd.DataFrame({
        'Keyword': keywords,
        'EDP': edp_values,
    })
    return edp_df


print("EDP helper functions defined.")

In [ ]:
# -----------------------------------------------------------------------
# Generate perturbed keywords (same for both models — deterministic)
# -----------------------------------------------------------------------
kw_list = fingerprint_store['keywords']
perturbed_keywords = generate_perturbations(kw_list, seed=SEED)

print(f"Generated {len(perturbed_keywords)} perturbations.")
print("Examples:")
for i in range(min(10, len(kw_list))):
    print(f"  '{kw_list[i]}' → '{perturbed_keywords[i]}'")

# -----------------------------------------------------------------------
# TARGET model — embed perturbed keywords and compute EDP
# -----------------------------------------------------------------------
print(f"\nEmbedding {len(perturbed_keywords)} perturbed keywords with TARGET model...")
target_perturbed_embs = embed_texts_with_seq_clf(
    perturbed_keywords, target_model, target_tokenizer, batch_size=64, max_len=32
)

# Use raw (un-normalised) embeddings for EDP — drift magnitude matters
# Re-embed originals in same order as kw_list (alphabetical)
target_orig_embs_aligned = np.array([
    target_kw_embs[_kw_to_idx[kw]] for kw in kw_list
], dtype=np.float32)

target_edp_df = compute_edp_scores(target_orig_embs_aligned, target_perturbed_embs, kw_list)
target_edp_df = target_edp_df.sort_values('EDP', ascending=False).reset_index(drop=True)

print(f"\nTarget EDP — top 10 most sensitive keywords:")
print(target_edp_df.head(10).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# REFERENCE model — embed perturbed keywords and compute EDP
# -----------------------------------------------------------------------
print(f"Embedding {len(perturbed_keywords)} perturbed keywords with REFERENCE model...")
base_perturbed_embs = embed_texts_with_gpt2model(
    perturbed_keywords, base_gpt2, base_tokenizer, batch_size=64, max_len=32
)

base_orig_embs_aligned = np.array([
    base_kw_embs[_kw_to_idx[kw]] for kw in kw_list
], dtype=np.float32)

base_edp_df = compute_edp_scores(base_orig_embs_aligned, base_perturbed_embs, kw_list)
base_edp_df = base_edp_df.sort_values('EDP', ascending=False).reset_index(drop=True)

print(f"\nReference EDP — top 10 most sensitive keywords:")
print(base_edp_df.head(10).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# Contrastive EDP  =  EDP_target / EDP_reference
# -----------------------------------------------------------------------
# Merge on keyword
cedp_df = target_edp_df[['Keyword', 'EDP']].merge(
    base_edp_df[['Keyword', 'EDP']],
    on='Keyword', suffixes=('_target', '_ref')
)
cedp_df['cEDP'] = cedp_df['EDP_target'] / cedp_df['EDP_ref'].clip(lower=1e-8)
cedp_df = cedp_df.sort_values('cEDP', ascending=False).reset_index(drop=True)

print("Contrastive EDP computed.")
print(f"  cEDP > 1: {(cedp_df['cEDP'] > 1).sum()} / {len(cedp_df)} keywords")
print(f"  mean cEDP = {cedp_df['cEDP'].mean():.4f}")
print(f"\nTop 10 keywords by cEDP (target more sensitive):")
print(cedp_df[['Keyword', 'EDP_target', 'EDP_ref', 'cEDP']].head(10).round(4).to_string(index=False))

# Assign dominant class for per-class visualisation
cedp_df['Dominant_Class'] = cedp_df['Keyword'].map(kw_to_class)

# Per-class heatmaps
for cls in pubmed_classes:
    cls_df = cedp_df[cedp_df['Dominant_Class'] == cls].head(TOP_K_DISPLAY)
    if len(cls_df) == 0:
        continue
    heatmap_data = cls_df.set_index('Keyword')[['cEDP']]

    fig, ax = plt.subplots(figsize=(3, max(4, len(cls_df) * 0.45)))
    sns.heatmap(
        heatmap_data, cmap="RdYlGn", center=1.0,
        annot=True, fmt=".2f", cbar=True, ax=ax,
    )
    ax.set_title(
        f"cEDP — Class: {cls} — top {TOP_K_DISPLAY}\n"
        f"(>1 = target more sensitive to perturbation)",
        fontsize=10,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Keyword")
    plt.tight_layout()
    plt.show()

# Distribution histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cedp_df['cEDP'].clip(upper=5), bins=40, color='coral', edgecolor='white', alpha=0.8)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='cEDP = 1 (neutral)')
ax.axvline(cedp_df['cEDP'].mean(), color='red', linestyle='--', label=f"mean={cedp_df['cEDP'].mean():.3f}")
ax.set_title("cEDP Distribution (clipped at 5)", fontsize=12)
ax.set_xlabel("cEDP = EDP_target / EDP_reference")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 12d. CCPG — Class-Conditional Prediction Gap

**Intuition**: If a model memorised the association between a keyword and a class, removing that keyword from class-specific sentences should cause a larger shift in classifier logits.

**Formula**:
$$\text{CCPG}(k, c) = \mathbb{E}_{x \in \text{sentences}(c, k)} \left[ \| \text{logits}(x) - \text{logits}(x_{\setminus k}) \|_2 \right]$$

where $x_{\setminus k}$ is the sentence with keyword $k$ removed.

**Contrastive**:
$$\text{cCCPG}(k, c) = \frac{\text{CCPG}_{\text{target}}(k, c)}{\text{CCPG}_{\text{ref}}(k, c)}$$

In [ ]:
import re

def find_keyword_sentences(df, keywords, text_col, label_id_col, id2label,
                           n_per_class=10, seed=SEED):
    """
    For each (keyword, class) pair, find up to n_per_class sentences from
    that class that contain the keyword (case-insensitive whole-word match).

    Returns
    -------
    sentences_dict : dict[(keyword, class_label)] → list of sentence strings
    """
    rng = np.random.default_rng(seed)
    sentences_dict = {}

    # Pre-lowercase text for faster matching
    texts_lower = df[text_col].str.lower().values
    labels = df[label_id_col].values
    texts = df[text_col].values

    for kw in tqdm(keywords, desc="Finding keyword sentences", leave=False):
        kw_lower = kw.lower()
        # Simple substring match (whole word via regex is too slow for ~1000 kws)
        mask = np.array([kw_lower in t for t in texts_lower])

        for class_id, class_label in id2label.items():
            class_mask = labels == class_id
            combined = mask & class_mask
            indices = np.where(combined)[0]

            if len(indices) == 0:
                continue
            if len(indices) > n_per_class:
                indices = rng.choice(indices, size=n_per_class, replace=False)

            sentences_dict[(kw, class_label)] = [texts[i] for i in indices]

    return sentences_dict


def compute_ccpg_scores(sentences_dict, keywords, model, tokenizer, id2label,
                        batch_size=32, max_len=MAX_LENGTH):
    """
    Class-Conditional Prediction Gap.

    For each (keyword, class) pair with available sentences:
      1. Get logits for original sentences
      2. Get logits for sentences with keyword removed
      3. CCPG = mean L2 norm of logit difference

    Returns
    -------
    ccpg_df : DataFrame with columns [Keyword, <class_label_1>, ..., <class_label_N>, Max_CCPG]
    """
    model.eval()
    class_labels = [id2label[i] for i in sorted(id2label.keys())]

    # Collect all (keyword, class) results
    results = {kw: {cls: np.nan for cls in class_labels} for kw in keywords}

    for (kw, cls_label), sents in tqdm(sentences_dict.items(), desc="CCPG", leave=False):
        if kw not in results:
            continue

        # Create masked versions: remove keyword (case-insensitive)
        pattern = re.compile(re.escape(kw), re.IGNORECASE)
        masked_sents = [pattern.sub('', s).strip() for s in sents]

        # Skip if all masked sentences are empty
        if all(len(s) == 0 for s in masked_sents):
            continue

        # Replace empty masked sentences with a single space
        masked_sents = [s if len(s) > 0 else ' ' for s in masked_sents]

        # Forward pass — original
        enc_orig = tokenizer(
            list(sents), padding=True, truncation=True,
            max_length=max_len, return_tensors='pt'
        )
        enc_orig = {k: v.to(device) for k, v in enc_orig.items()}

        # Forward pass — masked
        enc_mask = tokenizer(
            masked_sents, padding=True, truncation=True,
            max_length=max_len, return_tensors='pt'
        )
        enc_mask = {k: v.to(device) for k, v in enc_mask.items()}

        with torch.no_grad():
            logits_orig = model(**enc_orig).logits.cpu().float().numpy()
            logits_mask = model(**enc_mask).logits.cpu().float().numpy()

        # L2 norm of logit difference, averaged over sentences
        diff = np.linalg.norm(logits_orig - logits_mask, axis=1)
        results[kw][cls_label] = float(diff.mean())

    # Build DataFrame
    rows = []
    for kw in keywords:
        row = {'Keyword': kw}
        row.update(results[kw])
        rows.append(row)

    ccpg_df = pd.DataFrame(rows)
    ccpg_df['Max_CCPG'] = ccpg_df[class_labels].max(axis=1)
    ccpg_df = ccpg_df.sort_values('Max_CCPG', ascending=False).reset_index(drop=True)
    return ccpg_df


print("CCPG helper functions defined.")

In [ ]:
# -----------------------------------------------------------------------
# Find keyword-containing sentences (shared between CCPG and FIS)
# -----------------------------------------------------------------------
print("Finding keyword-containing sentences for each (keyword, class) pair...")
print(f"  Keywords: {len(kw_list)}, Classes: {PUBMED_NUM_LABELS}, n_per_class=10")

sentences_dict = find_keyword_sentences(
    train_df, kw_list, TEXT_COL, 'label_id', PUBMED_ID2LABEL,
    n_per_class=10, seed=SEED,
)

print(f"\nSentences found for {len(sentences_dict)} (keyword, class) pairs.")
# Show coverage stats
n_pairs_possible = len(kw_list) * PUBMED_NUM_LABELS
print(f"Coverage: {len(sentences_dict)} / {n_pairs_possible} possible pairs "
      f"({100 * len(sentences_dict) / n_pairs_possible:.1f}%)")

# How many sentences per pair on average
avg_sents = np.mean([len(v) for v in sentences_dict.values()])
print(f"Avg sentences per pair: {avg_sents:.1f}")

In [ ]:
# -----------------------------------------------------------------------
# TARGET model — CCPG scores
# -----------------------------------------------------------------------
print("Computing CCPG scores for TARGET model...")
print("  (this may take a few minutes — ~1000 keywords × ~10 sentences × 2 passes)")

target_ccpg_df = compute_ccpg_scores(
    sentences_dict, kw_list, target_model, target_tokenizer, PUBMED_ID2LABEL,
)

print(f"\nTarget CCPG computed. Shape: {target_ccpg_df.shape}")
print(f"\nTop 10 keywords by Max_CCPG — TARGET:")
print(target_ccpg_df[['Keyword', 'Max_CCPG']].head(10).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# REFERENCE model — CCPG scores
#
# The reference "classifier" is GPT2Model body + sklearn LogisticRegression.
# We compute logits as: embed(text) @ lr_probe.coef_.T + lr_probe.intercept_
# -----------------------------------------------------------------------
def compute_ccpg_scores_sklearn(sentences_dict, keywords, body_model, tokenizer,
                                lr_model, id2label, max_len=MAX_LENGTH):
    """
    CCPG using a frozen GPT2Model + sklearn LogisticRegression as the classifier.
    """
    body_model.eval()
    class_labels = [id2label[i] for i in sorted(id2label.keys())]
    coef = lr_model.coef_.astype(np.float32)           # (C, H)
    intercept = lr_model.intercept_.astype(np.float32)  # (C,)

    results = {kw: {cls: np.nan for cls in class_labels} for kw in keywords}

    for (kw, cls_label), sents in tqdm(sentences_dict.items(), desc="CCPG-ref", leave=False):
        if kw not in results:
            continue

        pattern = re.compile(re.escape(kw), re.IGNORECASE)
        masked_sents = [pattern.sub('', s).strip() for s in sents]
        if all(len(s) == 0 for s in masked_sents):
            continue
        masked_sents = [s if len(s) > 0 else ' ' for s in masked_sents]

        # Embed original
        embs_orig = embed_texts_with_gpt2model(
            list(sents), body_model, tokenizer, batch_size=32, max_len=max_len
        )
        # Embed masked
        embs_mask = embed_texts_with_gpt2model(
            masked_sents, body_model, tokenizer, batch_size=32, max_len=max_len
        )

        # Compute logits via LR coefficients
        logits_orig = embs_orig @ coef.T + intercept  # (N, C)
        logits_mask = embs_mask @ coef.T + intercept

        diff = np.linalg.norm(logits_orig - logits_mask, axis=1)
        results[kw][cls_label] = float(diff.mean())

    rows = []
    for kw in keywords:
        row = {'Keyword': kw}
        row.update(results[kw])
        rows.append(row)

    ccpg_df = pd.DataFrame(rows)
    ccpg_df['Max_CCPG'] = ccpg_df[class_labels].max(axis=1)
    ccpg_df = ccpg_df.sort_values('Max_CCPG', ascending=False).reset_index(drop=True)
    return ccpg_df


print("Computing CCPG scores for REFERENCE model...")
print("  (this may take a few minutes)")

base_ccpg_df = compute_ccpg_scores_sklearn(
    sentences_dict, kw_list, base_gpt2, base_tokenizer,
    lr_probe, PUBMED_ID2LABEL,
)

print(f"\nReference CCPG computed. Shape: {base_ccpg_df.shape}")
print(f"\nTop 10 keywords by Max_CCPG — REFERENCE:")
print(base_ccpg_df[['Keyword', 'Max_CCPG']].head(10).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# Contrastive CCPG  =  CCPG_target / CCPG_reference
# -----------------------------------------------------------------------
ccpg_class_cols = [c for c in target_ccpg_df.columns if c not in ('Keyword', 'Max_CCPG')]

cccpg_rows = []
for kw in kw_list:
    t_row = target_ccpg_df[target_ccpg_df['Keyword'] == kw]
    b_row = base_ccpg_df[base_ccpg_df['Keyword'] == kw]
    if len(t_row) == 0 or len(b_row) == 0:
        continue
    row = {'Keyword': kw}
    for cls in ccpg_class_cols:
        t_val = t_row[cls].values[0]
        b_val = b_row[cls].values[0]
        if np.isnan(t_val) or np.isnan(b_val) or b_val < 1e-8:
            row[f'cCCPG_{cls}'] = np.nan
        else:
            row[f'cCCPG_{cls}'] = t_val / b_val
    cccpg_rows.append(row)

cccpg_df = pd.DataFrame(cccpg_rows)
cccpg_cols = [f'cCCPG_{cls}' for cls in ccpg_class_cols]
cccpg_df['Max_cCCPG'] = cccpg_df[cccpg_cols].max(axis=1)
cccpg_df = cccpg_df.sort_values('Max_cCCPG', ascending=False).reset_index(drop=True)

print("Contrastive CCPG computed.")
print(f"\nTop 10 keywords by Max_cCCPG:")
print(cccpg_df[['Keyword', 'Max_cCCPG']].head(10).round(3).to_string(index=False))

# Assign dominant class
cccpg_df['Dominant_Class'] = cccpg_df['Keyword'].map(kw_to_class)

# Per-class heatmaps
for cls in pubmed_classes:
    col = f'cCCPG_{cls}'
    if col not in cccpg_df.columns:
        continue
    cls_df = cccpg_df[cccpg_df['Dominant_Class'] == cls].dropna(subset=[col]).head(TOP_K_DISPLAY)
    if len(cls_df) == 0:
        continue
    heatmap_data = cls_df.set_index('Keyword')[[col]]

    fig, ax = plt.subplots(figsize=(3, max(4, len(cls_df) * 0.45)))
    sns.heatmap(
        heatmap_data, cmap="RdYlGn", center=1.0,
        annot=True, fmt=".2f", cbar=True, ax=ax,
    )
    ax.set_title(
        f"cCCPG — Class: {cls} — top {TOP_K_DISPLAY}\n"
        f"(>1 = target more sensitive to keyword removal)",
        fontsize=10,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Keyword")
    plt.tight_layout()
    plt.show()

# Distribution histogram
all_cccpg_vals = cccpg_df[cccpg_cols].values.flatten()
all_cccpg_vals = all_cccpg_vals[~np.isnan(all_cccpg_vals)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(all_cccpg_vals, 0, 5), bins=40, color='mediumpurple', edgecolor='white', alpha=0.8)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='cCCPG = 1 (neutral)')
ax.axvline(np.nanmean(all_cccpg_vals), color='red', linestyle='--',
           label=f"mean={np.nanmean(all_cccpg_vals):.3f}")
ax.set_title("cCCPG Distribution (clipped at 5)", fontsize=12)
ax.set_xlabel("cCCPG = CCPG_target / CCPG_reference")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 12e. FIS — Fisher Information Score

**Intuition**: The gradient of the classification loss with respect to the input embedding measures how much the model's prediction depends on the exact embedding of a keyword. Higher gradient magnitude = steeper loss landscape = stronger memorisation.

**Formula**:
$$\text{FIS}(k, c) = \mathbb{E}_{x \in \text{sentences}(c, k)} \left[ \left\| \frac{\partial \mathcal{L}}{\partial \mathbf{e}(k)} \right\|^2 \right]$$

where $\mathcal{L}$ is the cross-entropy loss and $\mathbf{e}(k)$ is the input embedding of the keyword tokens.

**Contrastive**:
$$\text{cFIS}(k, c) = \frac{\text{FIS}_{\text{target}}(k, c)}{\text{FIS}_{\text{ref}}(k, c)}$$

In [ ]:
def compute_fis_scores(sentences_dict, keywords, model, tokenizer, id2label,
                       max_len=MAX_LENGTH):
    """
    Fisher Information Score — gradient-based sensitivity.

    For each (keyword, class) pair:
      1. Tokenize sentences containing the keyword
      2. Forward pass with embedding gradients enabled
      3. Compute cross-entropy loss using the class label
      4. Backprop to get gradients w.r.t. input embeddings
      5. FIS = mean ||grad||² across keyword token positions and sentences

    Returns
    -------
    fis_df : DataFrame with columns [Keyword, <class_labels...>, Max_FIS]
    """
    model.eval()
    class_labels = [id2label[i] for i in sorted(id2label.keys())]
    label2id = {v: k for k, v in id2label.items()}

    results = {kw: {cls: np.nan for cls in class_labels} for kw in keywords}
    loss_fn = torch.nn.CrossEntropyLoss(reduction='none')

    for (kw, cls_label), sents in tqdm(sentences_dict.items(), desc="FIS", leave=False):
        if kw not in results:
            continue

        class_id = label2id[cls_label]

        enc = tokenizer(
            list(sents), padding=True, truncation=True,
            max_length=max_len, return_tensors='pt'
        )
        input_ids = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)
        labels_tensor = torch.full((len(sents),), class_id, dtype=torch.long, device=device)

        # Get token IDs for the keyword
        kw_token_ids = set(tokenizer.encode(kw, add_special_tokens=False))

        # Use a hook to capture gradients on the embedding layer output
        embedding_grads = []
        embedding_outputs = []

        def fwd_hook(module, input, output):
            output.retain_grad()
            embedding_outputs.append(output)

        hook = model.transformer.wte.register_forward_hook(fwd_hook)

        try:
            # Standard forward pass — HuggingFace handles attention mask correctly
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits  # (B, C)

            loss = loss_fn(logits, labels_tensor).sum()
            loss.backward()

            # Extract gradient from captured embedding output
            grad = embedding_outputs[0].grad  # (B, T, H)

            fis_vals = []
            for b in range(len(sents)):
                for t in range(input_ids.size(1)):
                    if input_ids[b, t].item() in kw_token_ids and attention_mask[b, t].item() == 1:
                        g = grad[b, t]
                        fis_vals.append(float((g ** 2).sum().item()))

            if len(fis_vals) > 0:
                results[kw][cls_label] = float(np.mean(fis_vals))

        finally:
            hook.remove()
            embedding_grads.clear()
            embedding_outputs.clear()
            model.zero_grad()

    rows = []
    for kw in keywords:
        row = {'Keyword': kw}
        row.update(results[kw])
        rows.append(row)

    fis_df = pd.DataFrame(rows)
    fis_df['Max_FIS'] = fis_df[class_labels].max(axis=1)
    fis_df = fis_df.sort_values('Max_FIS', ascending=False).reset_index(drop=True)
    return fis_df


def compute_fis_scores_sklearn(sentences_dict, keywords, body_model, tokenizer,
                                lr_model, id2label, max_len=MAX_LENGTH):
    """
    FIS for the reference model (GPT2Model body + sklearn LR).

    Instead of model.score, we apply LR coefficients manually after getting
    embedding gradients from the body model.
    """
    body_model.eval()
    class_labels = [id2label[i] for i in sorted(id2label.keys())]
    label2id = {v: k for k, v in id2label.items()}

    coef_t = torch.tensor(lr_model.coef_, dtype=torch.float32, device=device)    # (C, H)
    intercept_t = torch.tensor(lr_model.intercept_, dtype=torch.float32, device=device)  # (C,)

    results = {kw: {cls: np.nan for cls in class_labels} for kw in keywords}
    loss_fn = torch.nn.CrossEntropyLoss(reduction='none')

    for (kw, cls_label), sents in tqdm(sentences_dict.items(), desc="FIS-ref", leave=False):
        if kw not in results:
            continue

        class_id = label2id[cls_label]

        enc = tokenizer(
            list(sents), padding=True, truncation=True,
            max_length=max_len, return_tensors='pt'
        )
        input_ids = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)
        labels_tensor = torch.full((len(sents),), class_id, dtype=torch.long, device=device)

        kw_token_ids = set(tokenizer.encode(kw, add_special_tokens=False))

        # Hook to capture embedding gradients
        embedding_outputs = []

        def fwd_hook(module, input, output):
            output.retain_grad()
            embedding_outputs.append(output)

        hook = body_model.wte.register_forward_hook(fwd_hook)

        try:
            outputs = body_model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden = outputs.last_hidden_state  # (B, T, H)

            # Last-token pooling
            seq_lens = attention_mask.sum(dim=1) - 1
            b_idx = torch.arange(last_hidden.size(0), device=device)
            pooled = last_hidden[b_idx, seq_lens]  # (B, H)

            # Apply LR coefficients as classification head
            logits = pooled @ coef_t.T + intercept_t  # (B, C)

            loss = loss_fn(logits, labels_tensor).sum()
            loss.backward()

            grad = embedding_outputs[0].grad  # (B, T, H)

            fis_vals = []
            for b in range(len(sents)):
                for t in range(input_ids.size(1)):
                    if input_ids[b, t].item() in kw_token_ids and attention_mask[b, t].item() == 1:
                        g = grad[b, t]
                        fis_vals.append(float((g ** 2).sum().item()))

            if len(fis_vals) > 0:
                results[kw][cls_label] = float(np.mean(fis_vals))

        finally:
            hook.remove()
            embedding_outputs.clear()
            body_model.zero_grad()

    rows = []
    for kw in keywords:
        row = {'Keyword': kw}
        row.update(results[kw])
        rows.append(row)

    fis_df = pd.DataFrame(rows)
    fis_df['Max_FIS'] = fis_df[class_labels].max(axis=1)
    fis_df = fis_df.sort_values('Max_FIS', ascending=False).reset_index(drop=True)
    return fis_df


print("FIS helper functions defined (target + reference variants).")

In [ ]:
# -----------------------------------------------------------------------
# TARGET model — FIS scores
# -----------------------------------------------------------------------
print("Computing FIS scores for TARGET model...")
print("  (this requires gradient computation — may take several minutes)")

target_fis_df = compute_fis_scores(
    sentences_dict, kw_list, target_model, target_tokenizer, PUBMED_ID2LABEL,
)

print(f"\nTarget FIS computed. Shape: {target_fis_df.shape}")
print(f"\nTop 10 keywords by Max_FIS — TARGET:")
print(target_fis_df[['Keyword', 'Max_FIS']].head(10).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# REFERENCE model — FIS scores
# -----------------------------------------------------------------------
print("Computing FIS scores for REFERENCE model...")

base_fis_df = compute_fis_scores_sklearn(
    sentences_dict, kw_list, base_gpt2, base_tokenizer,
    lr_probe, PUBMED_ID2LABEL,
)

print(f"\nReference FIS computed. Shape: {base_fis_df.shape}")
print(f"\nTop 10 keywords by Max_FIS — REFERENCE:")
print(base_fis_df[['Keyword', 'Max_FIS']].head(10).to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# Contrastive FIS  =  FIS_target / FIS_reference
# -----------------------------------------------------------------------
fis_class_cols = [c for c in target_fis_df.columns if c not in ('Keyword', 'Max_FIS')]

cfis_rows = []
for kw in kw_list:
    t_row = target_fis_df[target_fis_df['Keyword'] == kw]
    b_row = base_fis_df[base_fis_df['Keyword'] == kw]
    if len(t_row) == 0 or len(b_row) == 0:
        continue
    row = {'Keyword': kw}
    for cls in fis_class_cols:
        t_val = t_row[cls].values[0]
        b_val = b_row[cls].values[0]
        if np.isnan(t_val) or np.isnan(b_val) or b_val < 1e-12:
            row[f'cFIS_{cls}'] = np.nan
        else:
            row[f'cFIS_{cls}'] = t_val / b_val
    cfis_rows.append(row)

cfis_df = pd.DataFrame(cfis_rows)
cfis_cols = [f'cFIS_{cls}' for cls in fis_class_cols]
cfis_df['Max_cFIS'] = cfis_df[cfis_cols].max(axis=1)
cfis_df = cfis_df.sort_values('Max_cFIS', ascending=False).reset_index(drop=True)

print("Contrastive FIS computed.")
print(f"\nTop 10 keywords by Max_cFIS:")
print(cfis_df[['Keyword', 'Max_cFIS']].head(10).round(3).to_string(index=False))

# Assign dominant class
cfis_df['Dominant_Class'] = cfis_df['Keyword'].map(kw_to_class)

# Per-class heatmaps
for cls in pubmed_classes:
    col = f'cFIS_{cls}'
    if col not in cfis_df.columns:
        continue
    cls_df = cfis_df[cfis_df['Dominant_Class'] == cls].dropna(subset=[col]).head(TOP_K_DISPLAY)
    if len(cls_df) == 0:
        continue
    heatmap_data = cls_df.set_index('Keyword')[[col]]

    fig, ax = plt.subplots(figsize=(3, max(4, len(cls_df) * 0.45)))
    sns.heatmap(
        heatmap_data, cmap="RdYlGn", center=1.0,
        annot=True, fmt=".2f", cbar=True, ax=ax,
    )
    ax.set_title(
        f"cFIS — Class: {cls} — top {TOP_K_DISPLAY}\n"
        f"(>1 = target loss more sensitive to keyword)",
        fontsize=10,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Keyword")
    plt.tight_layout()
    plt.show()

# Distribution histogram
all_cfis_vals = cfis_df[cfis_cols].values.flatten()
all_cfis_vals = all_cfis_vals[~np.isnan(all_cfis_vals)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(all_cfis_vals, 0, 10), bins=40, color='goldenrod', edgecolor='white', alpha=0.8)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='cFIS = 1 (neutral)')
ax.axvline(np.nanmean(all_cfis_vals), color='red', linestyle='--',
           label=f"mean={np.nanmean(all_cfis_vals):.3f}")
ax.set_title("cFIS Distribution (clipped at 10)", fontsize=12)
ax.set_xlabel("cFIS = FIS_target / FIS_reference")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 12f. Cross-Metric Comparison

Compare all contrastive metrics (cEDP, cCCPG, cFIS) and NOS to identify keywords that show consistent membership signals across multiple metrics.

In [ ]:
# -----------------------------------------------------------------------
# Cross-Metric Comparison — combine all metric signals per keyword
# -----------------------------------------------------------------------

# Build a unified DataFrame with one row per keyword and columns for each metric
unified_rows = []
for kw in kw_list:
    row = {'Keyword': kw}

    # NOS (lower = more divergent = stronger signal)
    nos_row = nos_df[nos_df['Keyword'] == kw]
    row['NOS'] = nos_row['NOS'].values[0] if len(nos_row) else np.nan

    # cEDP (higher = more sensitive = stronger signal)
    cedp_row = cedp_df[cedp_df['Keyword'] == kw]
    row['cEDP'] = cedp_row['cEDP'].values[0] if len(cedp_row) else np.nan

    # Max cCCPG
    cccpg_row = cccpg_df[cccpg_df['Keyword'] == kw]
    row['Max_cCCPG'] = cccpg_row['Max_cCCPG'].values[0] if len(cccpg_row) else np.nan

    # Max cFIS
    cfis_row = cfis_df[cfis_df['Keyword'] == kw]
    row['Max_cFIS'] = cfis_row['Max_cFIS'].values[0] if len(cfis_row) else np.nan

    # Dominant class
    row['Dominant_Class'] = kw_to_class.get(kw, 'Unknown')

    unified_rows.append(row)

unified_df = pd.DataFrame(unified_rows)

# Composite membership signal score:
# Normalise each metric to [0, 1] range, invert NOS (low NOS = strong signal)
# Then average to get a composite score
from sklearn.preprocessing import MinMaxScaler

metric_cols = ['NOS', 'cEDP', 'Max_cCCPG', 'Max_cFIS']
scaler = MinMaxScaler()

# Fill NaN with neutral values before scaling
fill_vals = {'NOS': 1.0, 'cEDP': 1.0, 'Max_cCCPG': 1.0, 'Max_cFIS': 1.0}
unified_filled = unified_df[metric_cols].fillna(fill_vals)

# Invert NOS so that low NOS → high signal
unified_filled['NOS'] = 1.0 - unified_filled['NOS']

scaled = scaler.fit_transform(unified_filled)
unified_df['Composite_Signal'] = scaled.mean(axis=1)
unified_df = unified_df.sort_values('Composite_Signal', ascending=False).reset_index(drop=True)

print("Cross-Metric Comparison — Top 20 keywords by composite membership signal:")
print(unified_df[['Keyword', 'NOS', 'cEDP', 'Max_cCCPG', 'Max_cFIS',
                   'Composite_Signal', 'Dominant_Class']].head(20).round(3).to_string(index=False))

# --- Correlation matrix between metrics ---
corr_data = unified_df[['NOS', 'cEDP', 'Max_cCCPG', 'Max_cFIS']].copy()
corr_data['NOS'] = 1.0 - corr_data['NOS']  # invert for correlation
corr_matrix = corr_data.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap: metric correlation
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Spearman Correlation Between MIA Metrics\n(NOS inverted: high = strong signal)", fontsize=11)

# Heatmap: top 20 keywords × all metrics
top20 = unified_df.head(20).set_index('Keyword')[['NOS', 'cEDP', 'Max_cCCPG', 'Max_cFIS']]
# Invert NOS for display (low NOS = strong signal → show as high)
top20_display = top20.copy()
top20_display['1-NOS'] = 1.0 - top20_display['NOS']
top20_display = top20_display[['1-NOS', 'cEDP', 'Max_cCCPG', 'Max_cFIS']]

sns.heatmap(top20_display, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Top 20 Keywords — All Metrics\n(higher = stronger membership signal)", fontsize=11)
axes[1].set_ylabel("Keyword")

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# Update fingerprint_store with new metrics
# -----------------------------------------------------------------------
fingerprint_store['target']['nos_scores']   = nos_df
fingerprint_store['target']['edp_scores']   = target_edp_df
fingerprint_store['target']['ccpg_scores']  = target_ccpg_df
fingerprint_store['target']['fis_scores']   = target_fis_df

fingerprint_store['reference']['edp_scores']  = base_edp_df
fingerprint_store['reference']['ccpg_scores'] = base_ccpg_df
fingerprint_store['reference']['fis_scores']  = base_fis_df

fingerprint_store['contrastive'] = {
    'cedp':    cedp_df,
    'cccpg':   cccpg_df,
    'cfis':    cfis_df,
    'unified': unified_df,
}

print("fingerprint_store updated with new metrics:")
print(f"  target   → nos_scores, edp_scores, ccpg_scores, fis_scores")
print(f"  reference → edp_scores, ccpg_scores, fis_scores")
print(f"  contrastive → cedp, cccpg, cfis, unified")
print(f"  unified shape: {unified_df.shape}")

## 15. Visualisations — Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Plot 1: Bias score distributions ---
axes[0].hist(target_bias_vec, bins=30, alpha=0.6, label='Medical-GPT2 (Target)', color='tomato', edgecolor='white')
axes[0].hist(base_bias_vec,   bins=30, alpha=0.6, label='GPT-2 Base + Probe (Ref)', color='steelblue', edgecolor='white')
axes[0].set_title("BEE Bias Score Distribution", fontsize=12)
axes[0].set_xlabel("Bias Score (Max − Min sim)")
axes[0].set_ylabel("Count")
axes[0].legend()

# --- Plot 2: Scatter — target vs reference bias scores ---
axes[1].scatter(base_bias_vec, target_bias_vec, alpha=0.4, s=18, color='purple')
axes[1].set_xlabel("Reference (GPT-2 + Probe) Bias Score")
axes[1].set_ylabel("Target (Medical-GPT2) Bias Score")
axes[1].set_title(f"Bias Score Scatter\n(Pearson r = {pearson_r:.4f}, cos_sim = {cos_sim:.4f})", fontsize=11)

# Add diagonal reference line
lims = [min(base_bias_vec.min(), target_bias_vec.min()),
        max(base_bias_vec.max(), target_bias_vec.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1, alpha=0.5, label='y = x')
axes[1].legend()

# --- Plot 3: Side-by-side bar chart for top-K ---
top_common = target_bee_df.head(TOP_K_OVERLAP).copy()
base_score_map = dict(zip(base_bee_df['Keyword'], base_bee_df['Bias_Score']))
top_common['Ref_Score'] = top_common['Keyword'].map(lambda k: base_score_map.get(k, 0.0))

x     = np.arange(len(top_common))
width = 0.4
axes[2].barh(x - width/2, top_common['Bias_Score'], width,
             label='Medical-GPT2', color='tomato', alpha=0.85)
axes[2].barh(x + width/2, top_common['Ref_Score'],  width,
             label='GPT-2 + Probe', color='steelblue', alpha=0.85)
axes[2].set_yticks(x)
axes[2].set_yticklabels(top_common['Keyword'], fontsize=9)
axes[2].set_title(f"Top-{TOP_K_OVERLAP} Keywords by Target BEE Score", fontsize=11)
axes[2].set_xlabel("Bias Score")
axes[2].legend()

plt.suptitle("BEE Analysis — Medical-GPT2 vs GPT-2 Base Comparison",
             fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Bias-score heatmap for top-N shared & exclusive keywords
# --------------------------------------------------------------------------
N_HEATMAP = 25

# Combine and sort by target bias score
compare_df = pd.DataFrame({'Keyword': all_keywords_sorted,
                            'Target': target_bias_vec,
                            'Reference': base_bias_vec})
compare_df = compare_df.sort_values('Target', ascending=False).head(N_HEATMAP)

fig, ax = plt.subplots(figsize=(6, max(4, N_HEATMAP // 2)))
heat = compare_df.set_index('Keyword')[['Target', 'Reference']]
sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlOrRd",
            cbar_kws={'label': 'BEE Bias Score'}, ax=ax)
ax.set_title(f"Top-{N_HEATMAP} BEE Keywords by Target Score", fontsize=13)
ax.set_xlabel("Model")
ax.set_ylabel("Keyword")
plt.tight_layout()
plt.show()

## 16. Export Results to Excel

In [ ]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

OUTPUT_EXCEL = "bee_results_gpt2_pubmed.xlsx"

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:

    # Sheet 1 — Target model BEE (sorted by bias score)
    target_bee_df.to_excel(writer, sheet_name='Target_Medical-GPT2', index=False)

    # Sheet 2 — Reference model BEE (sorted by bias score)
    base_bee_df.to_excel(writer, sheet_name='Ref_GPT2-Probe', index=False)

    # Sheet 3 — Side-by-side comparison
    compare_all = pd.DataFrame({
        'Keyword': all_keywords_sorted,
        'Target_BiasScore': target_bias_vec,
        'Ref_BiasScore':    base_bias_vec,
        'Delta':            target_bias_vec - base_bias_vec,
    }).sort_values('Target_BiasScore', ascending=False).reset_index(drop=True)
    compare_all.to_excel(writer, sheet_name='Comparison', index=False)

    # Sheet 4 — NOS scores
    nos_df.to_excel(writer, sheet_name='NOS', index=False)

    # Sheet 5 — EDP (Target)
    target_edp_df.to_excel(writer, sheet_name='EDP_Target', index=False)

    # Sheet 6 — EDP (Reference)
    base_edp_df.to_excel(writer, sheet_name='EDP_Reference', index=False)

    # Sheet 7 — Contrastive EDP
    cedp_df.to_excel(writer, sheet_name='cEDP', index=False)

    # Sheet 8 — CCPG (Target)
    target_ccpg_df.to_excel(writer, sheet_name='CCPG_Target', index=False)

    # Sheet 9 — CCPG (Reference)
    base_ccpg_df.to_excel(writer, sheet_name='CCPG_Reference', index=False)

    # Sheet 10 — Contrastive CCPG
    cccpg_df.to_excel(writer, sheet_name='cCCPG', index=False)

    # Sheet 11 — FIS (Target)
    target_fis_df.to_excel(writer, sheet_name='FIS_Target', index=False)

    # Sheet 12 — FIS (Reference)
    base_fis_df.to_excel(writer, sheet_name='FIS_Reference', index=False)

    # Sheet 13 — Contrastive FIS
    cfis_df.to_excel(writer, sheet_name='cFIS', index=False)

    # Sheet 14 — Unified cross-metric comparison
    unified_df.to_excel(writer, sheet_name='Unified_Metrics', index=False)

    # Sheet 15 — Summary metrics
    summary_data = {
        'Metric': [
            'Dataset',
            'Target model',
            'Target num_labels',
            'Target label names',
            'Reference model',
            'Reference num_labels',
            'Reference label names',
            'Num keywords',
            'Cosine similarity (bias vectors)',
            'Pearson correlation (bias vectors)',
            f'Jaccard top-{TOP_K_OVERLAP}',
            'Mean NOS',
            'Mean cEDP',
            'Mean cCCPG (non-NaN)',
            'Mean cFIS (non-NaN)',
            'SEED',
        ],
        'Value': [
            DATASET_NAME,
            MODEL_TARGET_ID,
            target_model.config.num_labels,
            str(TARGET_LABEL_NAMES),
            MODEL_BASE_ID,
            PUBMED_NUM_LABELS,
            str(list(PUBMED_ID2LABEL.values())),
            len(clean_keywords),
            f"{cos_sim:.4f}",
            f"{pearson_r:.4f}",
            f"{jaccard:.4f}",
            f"{nos_df['NOS'].mean():.4f}",
            f"{cedp_df['cEDP'].mean():.4f}",
            f"{cccpg_df['Max_cCCPG'].mean():.4f}",
            f"{cfis_df['Max_cFIS'].mean():.4f}",
            SEED,
        ],
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

print(f"Results exported to: {OUTPUT_EXCEL}")
print(f"  Sheets: {15} (BEE + NOS + EDP + CCPG + FIS + Unified + Summary)")

## 17. Final Summary

In [ ]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"""
Dataset : {DATASET_NAME}
  Train samples used : {len(train_df):,}
  Classes            : {PUBMED_NUM_LABELS} → {list(PUBMED_ID2LABEL.values())}

TARGET model : {MODEL_TARGET_ID}
  Classes     : {target_model.config.num_labels} → {TARGET_LABEL_NAMES}
  Training    : none (used as-is)

REFERENCE model : {MODEL_BASE_ID}
  Body        : fully frozen GPT-2
  Probe       : sklearn LogisticRegression on {N_PROBE_SAMPLES:,} frozen embeddings
  Classes     : {PUBMED_NUM_LABELS} → {list(PUBMED_ID2LABEL.values())}
  Training    : none (sklearn fit on frozen features, no PyTorch gradients)

BEE Analysis
  Keywords extracted (YAKE)          : {len(clean_keywords)}
  Cosine similarity (bias vectors)   : {cos_sim:.4f}
  Pearson correlation (bias vectors) : {pearson_r:.4f}
  Jaccard top-{TOP_K_OVERLAP} keywords               : {jaccard:.4f}

Novel MIA Metrics
  NOS  (Neighborhood Overlap Score)
    Mean NOS                         : {nos_df['NOS'].mean():.4f}
    Std NOS                          : {nos_df['NOS'].std():.4f}
    Keywords with NOS < 0.5          : {(nos_df['NOS'] < 0.5).sum()}

  EDP  (Embedding Drift under Perturbation)
    Mean cEDP (target/ref ratio)     : {cedp_df['cEDP'].mean():.4f}
    Keywords with cEDP > 1           : {(cedp_df['cEDP'] > 1).sum()} / {len(cedp_df)}

  CCPG (Class-Conditional Prediction Gap)
    Mean Max_cCCPG                   : {cccpg_df['Max_cCCPG'].mean():.4f}
    Keywords with Max_cCCPG > 1      : {(cccpg_df['Max_cCCPG'] > 1).sum()} / {len(cccpg_df)}

  FIS  (Fisher Information Score)
    Mean Max_cFIS                    : {cfis_df['Max_cFIS'].mean():.4f}
    Keywords with Max_cFIS > 1       : {(cfis_df['Max_cFIS'] > 1).sum()} / {len(cfis_df)}

  Composite Signal (top 5 keywords)  :
""")

for _, row in unified_df.head(5).iterrows():
    print(f"    {row['Keyword']:<20s}  signal={row['Composite_Signal']:.3f}  class={row['Dominant_Class']}")

print(f"""
Reproducibility
  SEED = {SEED}  (set for random, numpy, torch, cuda, sklearn, YAKE)
""")

print("=" * 70)

---
### Interpretation Guide

| Metric | High value | Low value |
|---|---|---|
| **Cosine similarity** | Target fingerprint matches PubMed probe | Different biased vocabulary |
| **Pearson r** | Strong linear agreement in keyword biases | Uncorrelated biases |
| **Jaccard top-K** | Same keywords are most biased in both models | Different top keywords |
| **NOS** | Identical keyword neighborhoods (no signal) | Divergent neighborhoods (membership signal) |
| **cEDP** | Target more sensitive to typos (memorisation) | Reference more sensitive |
| **cCCPG** | Target relies more on keyword for classification | Reference relies more |
| **cFIS** | Target loss more sensitive to keyword embedding | Reference more sensitive |
| **Composite Signal** | Multiple metrics agree on strong membership signal | Weak or conflicting signals |